# 05.5 — Checkpoint & resume state machine

A 500-epoch training run can be killed at any moment — a preempted cluster node, a crash, a `Ctrl+C`. Checkpointing lets it pick up where it left off. But there's a subtlety this project is careful about: it keeps **two** checkpoints — *Current* (the latest, for resuming) and *Optimal* (the best-so-far, for final results) — and resume reads Current, never Optimal (Critical Note #2). This notebook is about that state machine.

This notebook covers:

* `state_dict` save/load — the checkpoint anatomy ([02.6 §2.4](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb) recap).
* The Current vs Optimal distinction and why both exist.
* The resume decision: detect an existing Current, restore, continue.
* Why the optimizer state is deliberately NOT saved (Critical Note #3).

**Prerequisites:** [02.6 nn.Module](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb) (state_dict), [05.1 the loop](05.1_the_custom_training_loop.ipynb).

## Section 1 — What MATLAB does

The MATLAB pipeline (`cgg_trainNetwork`) saves two networks with the same two roles:

```matlab
save(CurrentPath, 'net', 'epoch', 'iteration');           % every epoch — for resume
if validationMetric > bestMetric
    bestMetric = validationMetric;
    save(OptimalPath, 'net');                              % only on improvement — for results
end
% On startup:
if isfile(CurrentPath)
    load(CurrentPath); startEpoch = epoch + 1;             % RESUME reads Current
end
```

Two files, two purposes: **Current** is overwritten every epoch so resume always has the latest; **Optimal** is written only when validation improves, so it holds the best model for the final CM-table. The Python port keeps this exactly — down to reading Current (never Optimal) on resume.

## Section 2 — The Python concepts you need

### 2.1 — A checkpoint is a saved `state_dict` (plus bookkeeping)

[02.6 §2.4](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb) showed `torch.save(model.state_dict(), path)`. A resume checkpoint bundles that with the *where-was-I* metadata — epoch, iteration, best metric:

In [ ]:
import torch
from torch import nn
from pathlib import Path
import tempfile

model = nn.Linear(8, 3)
ckpt_dir = Path(tempfile.mkdtemp())

# A resume checkpoint = weights + progress bookkeeping:
torch.save({
    "model_state_dict": model.state_dict(),
    "epoch": 42,
    "iteration": 3400,
    "best_metric": 0.87,
}, ckpt_dir / "current_state.pt")

loaded = torch.load(ckpt_dir / "current_state.pt", weights_only=True)
print("restored: epoch", loaded["epoch"], "| best_metric", loaded["best_metric"])

The weights answer "what has the model learned"; the bookkeeping answers "which epoch do I resume at" (`epoch + 1`) and "what's the bar an Optimal save must beat" (`best_metric`).

### 2.2 — The two-checkpoint state machine

The rule, drawn as a decision each epoch:

```
      ┌─────────────── every epoch ───────────────┐
      │  train one epoch                           │
      │  save CURRENT  (overwrite — latest wins)   │  ← resume reads THIS
      │  validate                                  │
      │  if val_metric > best:                     │
      │      best = val_metric                     │
      │      save OPTIMAL  (overwrite — best wins)  │  ← final results read THIS
      └────────────────────────────────────────────┘
```

Current changes every epoch; Optimal changes only on improvement. Late in training a model can *overfit* — its Current (latest) weights may be worse than its Optimal (best-validation) weights. That's the whole reason for two files: **resume needs the latest state to continue correctly, but the deliverable needs the best state.** Confusing them — resuming from Optimal — would silently rewind training to an earlier point every restart.

### 2.3 — The project's checkpoint functions, live

In [ ]:
from neural_data_decoding.training.checkpoint import (
    save_current_checkpoint, save_optimal_checkpoint,
    load_current_checkpoint, has_existing_checkpoint,
    CURRENT_CHECKPOINT_FILENAME, OPTIMAL_CHECKPOINT_FILENAME,
)

fresh = Path(tempfile.mkdtemp())
print("before any save, has_existing_checkpoint:", has_existing_checkpoint(fresh))

model = nn.Linear(8, 3)
save_current_checkpoint(fresh, model=model, epoch=5, iteration=200, best_metric=0.7)
save_optimal_checkpoint(fresh, model=model, epoch=5, metric=0.7)
print("after saves, files:", sorted(p.name for p in fresh.glob("*.pt")))
print("has_existing_checkpoint:", has_existing_checkpoint(fresh))

In [ ]:
# Resume reads CURRENT and returns the bookkeeping to continue from:
model2 = nn.Linear(8, 3)               # a fresh model...
state = load_current_checkpoint(fresh, model=model2)   # ...gets the saved weights in-place
print(f"resumed at epoch {state.epoch} → training continues at epoch {state.epoch + 1}")
print(f"best metric to beat: {state.best_metric}")

`load_current_checkpoint` loads the weights *into* `model2` (in place) and returns a `CheckpointState` with the progress fields. The loop resumes at `state.epoch + 1` and carries `state.best_metric` forward as the bar for the next Optimal save. `has_existing_checkpoint` is the startup probe: present → resume; absent → fresh run.

### 2.4 — The optimizer state is deliberately NOT saved

Here's a design decision worth understanding. Adam/AdamW carry internal momentum state ([02.7 §2.1](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb)) — and a *fully* correct resume would restore it. This project (matching MATLAB, Critical Note #3) **does not**: the checkpoint holds only model weights, so after a resume the optimizer restarts its moments from zero. The tradeoff: smaller checkpoints (optimizer state can double the file size) at the cost of a brief warmup as the moments rebuild over the first few iterations. For a 500-epoch run resumed rarely, that warmup is negligible — a deliberate simplicity-over-purity choice.

## Section 3 — The `neural_data_decoding` implementation

`save_current_checkpoint` — weights + the three bookkeeping fields, no optimizer:

In [ ]:
from pathlib import Path as P
src = P("../../src/neural_data_decoding/training/checkpoint.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "torch.save(" in line and "model_state_dict" in "".join(src[j:j+3]))
for k in range(i, min(i + 11, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The saved dict has `model_state_dict` + `epoch` + `iteration` + `best_metric` — and NO `optimizer_state_dict` (§2.4, Critical Note #3).
* Two filename constants (`current_state.pt`, `optimal_state.pt`) keep the two roles unambiguous — no stringly-typed paths scattered around.
* `fit_supervised`'s docstring spells out the resume contract: "Resume reads Current, never Optimal" and "The optimizer is not restored" — the state machine documented at the entry point.
* `save_optimal_checkpoint` is gated on the improvement check in the loop; `save_current_checkpoint` runs unconditionally each epoch — §2.2's two cadences.

## Section 4 — Hands-on exercises

### Exercise 4.1 — simulate the state machine

Run a fake 6-epoch loop with random validation metrics; save Current every epoch and Optimal only on improvement. Track which epoch's weights each file holds at the end.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
d = Path(tempfile.mkdtemp())
m = nn.Linear(4, 2)
best = -1.0
metrics = [0.5, 0.7, 0.6, 0.8, 0.75, 0.72]      # peaks at epoch 3
for epoch, val in enumerate(metrics):
    save_current_checkpoint(d, model=m, epoch=epoch, iteration=epoch*10, best_metric=best)
    if val > best:
        best = val
        save_optimal_checkpoint(d, model=m, epoch=epoch, metric=best)
        print(f"  epoch {epoch}: val {val} > best → Optimal updated")
cur = torch.load(d / CURRENT_CHECKPOINT_FILENAME, weights_only=True)
opt = torch.load(d / OPTIMAL_CHECKPOINT_FILENAME, weights_only=True)
print(f"CURRENT holds epoch {cur['epoch']} (latest); OPTIMAL holds epoch {opt['epoch']} (best val)")

### Exercise 4.2 — the resume/fresh decision

Write the startup logic: given a checkpoint dir, decide `start_epoch` — resume at `state.epoch + 1` if a Current exists, else 0.

In [ ]:
# Reveal:
def decide_start(checkpoint_dir, model):
    if has_existing_checkpoint(checkpoint_dir):
        state = load_current_checkpoint(checkpoint_dir, model=model)
        return state.epoch + 1
    return 0

print("resume dir → start_epoch:", decide_start(d, nn.Linear(4, 2)))      # epoch 6 (after the 6-epoch run)
print("fresh dir  → start_epoch:", decide_start(Path(tempfile.mkdtemp()), nn.Linear(4, 2)))

## Section 5 — Common errors

### Resume rewinds training every restart

You loaded Optimal instead of Current — Optimal is the best-*validation* model, which may be many epochs behind the latest. Resume ALWAYS reads Current (§2.2, Critical Note #2).

### `RuntimeError: Error(s) in loading state_dict`

Architecture drift between save and load ([02.6 §5](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb)) — the model you're loading into differs from the one saved. The keys/shapes must match exactly.

### Final results use the wrong weights

The deliverable (CM-table) must come from Optimal, not Current — the latest weights can be overfit. This project writes the CM-tables from the Optimal state precisely to avoid this.

### First few iterations after resume look shaky

Expected (§2.4) — the optimizer's momentum restarts from zero, so early post-resume steps are noisier until the moments rebuild. Not a bug; the deliberate optimizer-state-not-saved tradeoff.

### `has_existing_checkpoint` false on a dir that has files

It checks for the specific `current_state.pt`/`optimal_state.pt` names — other `.pt` files don't count. A partial/renamed checkpoint won't be detected as resumable.

## Section 6 — Further reading

- [Saving & loading a general checkpoint](https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html) — including the optimizer-state case this project skips.
- [`src/neural_data_decoding/training/checkpoint.py`](../../src/neural_data_decoding/training/checkpoint.py) — the two-file state machine.
- [`fit_supervised` resume semantics](../../src/neural_data_decoding/training/lifecycle.py) — the docstring stating "Resume reads Current, never Optimal."

Next up: **[05.6 the two-stage lifecycle](05.6_the_two_stage_lifecycle.ipynb)** — unsupervised pre-training → supervised fine-tuning, and the weight handoff between them.